In [ ]:
import agama
import matplotlib.pyplot as plt
import numpy as np

agama.setUnits(mass=1, length=1, velocity=1)
units = agama.getUnits()


# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

SMALL_SIZE = 10
MEDIUM_SIZE = 10
BIGGER_SIZE = 10

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title


In [ ]:
lz_scale = 1e3
et_scale = 1e5

In [ ]:
sim_dir = "/Users/z5114326/Documents/simulations/"
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]

In [ ]:
# mw_fil = "../.venv312/lib/python3.12/site-packages/agama/data/McMillan17.ini"
mw_fil = "../.venv312/lib/python3.12/site-packages/agama/data/PriceWhelan22.ini"
mw_pot = agama.Potential(mw_fil)
mw_af = agama.ActionFinder(mw_pot, interp=False)

rs = np.arange(-100, 100, 0.5)
ps = np.column_stack((rs, rs * 0, rs * 0))

vc_m = np.sqrt(-rs * mw_pot.force(ps)[:, 0])
v_m = np.column_stack((vc_m * 0, vc_m, vc_m * 0))
pv_m = np.concatenate((ps, v_m), axis=1)
lz_m = mw_af(pv_m)[:, 2]
et_m = 0.5 * np.sum(np.array(v_m) ** 2, axis=1) + mw_pot.potential(ps)

lz_mw = np.arange(-10, 10, 0.01)
et_mw = []
for lz in lz_mw:
    if lz < -0.58:
        et_mw.append(-1.316)
    elif lz < 0.58:
        et_mw.append(-1.416 + 0.3 * lz**2)
    else:
        et_mw.append(-1.341 + 0.075 * lz**2)
et_mw = np.asarray(et_mw)

Rlim = mw_pot.Rcirc(E=-1.316 * et_scale)
p_lim = np.column_stack((Rlim, Rlim * 0, Rlim * 0))
vc_lim = np.sqrt(-Rlim * mw_pot.force(p_lim)[:, 0])
v_lim = np.column_stack((vc_lim * 0, vc_lim, vc_lim * 0))
pv_lim = np.concatenate((p_lim, v_lim), axis=1)
lz_lim = mw_af(pv_lim)[:, 2] / lz_scale

lz_msk = -lz_lim <= lz_mw
et_msk = et_mw <= 0

lz_mw = lz_mw[lz_msk & et_msk]
et_mw = et_mw[lz_msk & et_msk]


Rc_mw = mw_pot.Rcirc(E=(et_mw * et_scale))
pc_mw = np.column_stack((Rc_mw, Rc_mw * 0, Rc_mw * 0))
vc_mw = np.sqrt(-Rc_mw * mw_pot.force(pc_mw)[:, 0])
v_mw = np.column_stack((vc_mw * 0, vc_mw, vc_mw * 0))
pv_mw = np.concatenate((pc_mw, v_mw), axis=1)
lz_cmw = mw_af(pv_mw)[:, 2]

circ = (lz_mw * lz_scale) / lz_cmw

Et0_mw = mw_pot.potential([0, 0, 0])
eps = (et_mw * et_scale) / np.abs(Et0_mw)

In [ ]:
for sim in sim_lst:
    pot_file = sim_dir + sim + "/potentials/snap_%d/combined_snap_%d.ini" % (600, 600)
    pot_fire = agama.Potential(pot_file)
    af_fire = agama.ActionFinder(pot_fire, interp=False)

    vc_f = np.sqrt(-rs * pot_fire.force(ps)[:, 0])
    v_f = np.column_stack((vc_f * 0, vc_f, vc_f * 0))
    pv_f = np.concatenate((ps, v_f), axis=1)
    lz_f = af_fire(pv_f)[:, 2]
    et_f = 0.5 * np.sum(np.array(v_f) ** 2, axis=1) + pot_fire.potential(ps)

    Et0_f = pot_fire.potential([0, 0, 0])
    et_fb = (eps * np.abs(Et0_f)) / et_scale

    Rc_fb = pot_fire.Rcirc(E=(et_fb * et_scale))
    pc_fb = np.column_stack((Rc_fb, Rc_fb * 0, Rc_fb * 0))
    vc_fb = np.sqrt(-Rc_fb * pot_fire.force(pc_fb)[:, 0])
    v_fb = np.column_stack((vc_fb * 0, vc_fb, vc_fb * 0))
    pv_fb = np.concatenate((pc_fb, v_fb), axis=1)
    lz_cfb = af_fire(pv_fb)[:, 2]

    lz_fb = (circ * lz_cfb) / lz_scale

    plt.figure()

    plt.plot(lz_m / lz_scale, et_m / et_scale, c="g")
    plt.plot(lz_mw, et_mw, c="g", ls="--")

    plt.plot(lz_f / lz_scale, et_f / et_scale, c="r")
    plt.plot(lz_fb, et_fb, c="r", ls="--")

    plt.xlabel(r"L$_{z}$ (10$^3$ kpc km s$^{-1}$)")
    plt.ylabel(r"E (10$^5$ km$^{2}$ s$^{-2}$)")

    plt.show()